# Finding pairs with unsupervised learning
### EUTIC Quant final project: PCA + clustering for pairs selection

Based on Sarmento & Horta, *Enhancing a Pairs Trading strategy with the application of Machine Learning* (Section III).

**The problem.** The pairs trading notebook showed that KO/PEP, the textbook pair, fails the cointegration test and has no edge left: everyone trades it. To find pairs other people haven't, we have two naive options:

| Search method | Problem |
|---|---|
| Test **every** possible pair | With $N$ assets there are $N(N-1)/2$ pairs. At a 5% significance level, about 1 in 20 unrelated pairs passes by luck: **spurious pairs**. |
| Test only pairs in the **same sector** | Fewer false positives, but we only find the obvious pairs everyone else finds. |

**The idea.** Let the data decide which assets belong together:

1. **PCA**: describe each ETF by a handful of numbers (how it reacts to the main hidden "risk factors").
2. **Clustering**: group ETFs whose descriptions are similar. Only search for pairs *inside* a cluster.
3. **Filters**: keep only pairs that are cointegrated, mean-reverting, revert at a tradeable speed, and cross their mean often.

We compare four ways of searching: **all pairs**, **same category**, **K-means clusters** and **OPTICS clusters**.

**Today (Sunday): run Parts 0 to 4.** Part 5 (the filters) is for Tuesday.

## Part 0. Parameters

Everything you might want to change lives in this cell.

**Time windows** (same structure as the paper):
- **Formation** 2020 to 2022: we find clusters and pairs using *only* this data.
- **Validation** 2023 and **Test** 2024: used on Friday for the backtest. Nothing in this notebook looks at them, which is how we avoid **look-ahead bias**.

**Universe.** Commodity-linked ETFs, each labelled with a category (like the paper's 5 categories), plus a few "outsiders" that are *not* commodity funds. If clustering works, the outsiders should mostly end up as noise or in their own group.

Any ticker that doesn't exist, has gaps, or is too illiquid is dropped automatically in Part 1, so you don't need to check each one by hand.

In [1]:
import os
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, OPTICS
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE

# ---- Time windows ----
DATA_START, DATA_END = "2020-01-01", "2025-01-01"      # yfinance end date is exclusive
FORMATION = ("2020-01-01", "2022-12-31")               # clusters + pairs found here only

# ---- Universe: ticker -> category ----
UNIVERSE = {
    # Energy
    "XLE": "Energy", "VDE": "Energy", "XOP": "Energy", "OIH": "Energy", "FCG": "Energy",
    "IXC": "Energy", "IEO": "Energy", "XES": "Energy", "IEZ": "Energy",
    "USO": "Energy", "BNO": "Energy", "UNG": "Energy", "AMLP": "Energy",
    "URA": "Energy", "URNM": "Energy",
    # Precious metals (GLD/IAU and SLV/SIVR are deliberate duplicates: a sanity check)
    "GLD": "Precious metals", "IAU": "Precious metals", "SLV": "Precious metals",
    "SIVR": "Precious metals", "PPLT": "Precious metals", "PALL": "Precious metals",
    "GDX": "Precious metals", "GDXJ": "Precious metals", "SIL": "Precious metals",
    "RING": "Precious metals",
    # Industrial metals
    "XME": "Industrial metals", "COPX": "Industrial metals", "CPER": "Industrial metals",
    "PICK": "Industrial metals", "LIT": "Industrial metals", "REMX": "Industrial metals",
    "DBB": "Industrial metals", "SLX": "Industrial metals",
    # Agriculture
    "DBA": "Agriculture", "CORN": "Agriculture", "WEAT": "Agriculture",
    "SOYB": "Agriculture", "MOO": "Agriculture",
    # Broad commodities
    "DBC": "Broad commodities", "GSG": "Broad commodities", "PDBC": "Broad commodities",
    "COMT": "Broad commodities",
    # Outsiders: not commodity funds
    "SPY": "Outsider", "TLT": "Outsider", "XLK": "Outsider", "XLU": "Outsider",
    "XLV": "Outsider", "XLB": "Outsider",
}

MIN_DOLLAR_VOLUME = 1e6     # average daily $ traded in the formation window
MAX_MISSING_DAYS = 5        # tolerate a few gaps (filled forward), drop anything worse

N_COMPONENTS = 5            # PCA dimensions per ETF (the paper uses 5)
OPTICS_MIN_SAMPLES = 3      # smallest group OPTICS will call a cluster
RANDOM_STATE = 0            # makes K-means and t-SNE reproducible

# Colours: one fixed colour per category, used in every chart
CATEGORY_COLORS = {
    "Energy": "#2a78d6", "Precious metals": "#eb6834", "Industrial metals": "#1baf7a",
    "Agriculture": "#eda100", "Broad commodities": "#e87ba4", "Outsider": "#6250d6",
}
NOISE_COLOR = "#b4b2a9"

print(f"{len(UNIVERSE)} tickers in the starting universe")

48 tickers in the starting universe


## Part 1. Data

We download daily adjusted prices and volumes once and **save them to CSV**. yfinance is an unofficial API and can block you if you download repeatedly, and a saved file means both of you work from identical data. To re-download (e.g. after changing the universe), delete the two CSV files.

Then we clean the universe:
1. **Gaps.** Drop ETFs missing more than a few days anywhere in 2020 to 2024 (usually: launched too late, or closed). The backtest on Friday needs them to exist in the test year too.
2. **Liquidity.** Drop ETFs that trade less than `MIN_DOLLAR_VOLUME` per day. Illiquid funds are expensive to trade, so a backtest on them would be too optimistic.

In [2]:
PRICE_FILE, VOLUME_FILE = "etf_prices.csv", "etf_volumes.csv"

if os.path.exists(PRICE_FILE) and os.path.exists(VOLUME_FILE):
    prices = pd.read_csv(PRICE_FILE, index_col=0, parse_dates=True)
    volumes = pd.read_csv(VOLUME_FILE, index_col=0, parse_dates=True)
    print("Loaded saved data.")
else:
    raw = yf.download(list(UNIVERSE), start=DATA_START, end=DATA_END,
                      auto_adjust=True, progress=False)
    prices, volumes = raw["Close"], raw["Volume"]
    prices.to_csv(PRICE_FILE); volumes.to_csv(VOLUME_FILE)
    print("Downloaded and saved data.")

print(f"Shape: {prices.shape[0]} days x {prices.shape[1]} ETFs")

Loaded saved data.
Shape: 1258 days x 48 ETFs


In [3]:
# 1. Gaps: count missing days per ETF over the whole period
missing = prices.isna().sum()
too_gappy = missing[missing > MAX_MISSING_DAYS].index.tolist()

# 2. Liquidity: average daily dollar volume in the formation window
form_slice = slice(*FORMATION)
dollar_vol = (prices.loc[form_slice] * volumes.loc[form_slice]).mean()
too_illiquid = dollar_vol[dollar_vol < MIN_DOLLAR_VOLUME].index.difference(too_gappy).tolist()

keep = [t for t in prices.columns if t not in too_gappy + too_illiquid]
prices = prices[keep].ffill().dropna()        # fill the few small gaps forward

print(f"Dropped for gaps ({len(too_gappy)}):      {too_gappy}")
print(f"Dropped as illiquid ({len(too_illiquid)}): {too_illiquid}")
print(f"\nKept {len(keep)} ETFs:")
categories = pd.Series({t: UNIVERSE[t] for t in keep}, name="category")
print(categories.value_counts().to_string())

Dropped for gaps (0):      []
Dropped as illiquid (0): []

Kept 48 ETFs:
category
Energy               15
Precious metals      10
Industrial metals     8
Outsider              6
Agriculture           5
Broad commodities     4


## Part 2. PCA: a short description of each ETF

**Why returns, not prices?** Almost every price series trends upwards or downwards over three years, so two unrelated ETFs can look "similar" just because both trended. Daily returns, $R_{i,t} = \frac{P_{i,t} - P_{i,t-1}}{P_{i,t-1}}$, remove the trend and leave how each ETF actually *moves* day to day.

**What PCA does here.** Our returns form a table with ~750 rows (days) and ~45 columns (ETFs). PCA finds a small number of hidden patterns, the **principal components**, that explain most of the day-to-day movement across all ETFs at once. You can think of each one as a **risk factor**:
- Component 1 is usually "the whole market / all commodities moved together today".
- Later components are things like "oil up, gold down" or "metals vs everything else".

Each ETF gets a **loading** on each component: how strongly it reacts to that factor. With 5 components, each ETF is now described by **5 numbers** instead of 750 returns. Two ETFs with similar loadings react to the world in similar ways, which is exactly what we want a pair to do.

**Why only 5?** Clustering works badly in many dimensions (the *curse of dimensionality*: in high dimensions, every point ends up roughly equally far from every other). The paper keeps it below 15 and found 5 worked as well as anything higher.

We **standardise** each ETF's returns first (mean 0, standard deviation 1), so a very volatile ETF doesn't dominate the components just because its numbers are bigger.

In [ ]:
returns = prices.loc[form_slice].pct_change().dropna()        # formation window only
returns_std = StandardScaler().fit_transform(returns)         # each ETF: mean 0, std 1

pca = PCA(n_components=N_COMPONENTS).fit(returns_std)

# Each ETF's loadings on the components: one row per ETF, one column per component
features = pd.DataFrame(pca.components_.T, index=returns.columns,
                        columns=[f"PC{k+1}" for k in range(N_COMPONENTS)])

explained = pca.explained_variance_ratio_
print("Share of daily movement explained by each component:")
for k, e in enumerate(explained):
    print(f"  PC{k+1}: {e:6.1%}")
print(f"  Total: {explained.sum():6.1%}")

features.round(3).head(8)

In [ ]:
# How much does each extra component add? (shows why a few components are enough)
full = PCA().fit(returns_std).explained_variance_ratio_[:15]

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(range(1, len(full) + 1), full, color="#2a78d6", width=0.6)
ax.axvline(N_COMPONENTS + 0.5, color="#5f5e5a", ls=":", lw=1)
ax.text(N_COMPONENTS + 0.7, full.max() * 0.9, f"we keep {N_COMPONENTS}", color="#5f5e5a")
ax.set_xlabel("Principal component"); ax.set_ylabel("Share of variance explained")
ax.set_title("PC1 (the common market factor) dominates; later components add little")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## Part 3. Clustering

We now have one point per ETF in 5-dimensional space. Clustering groups nearby points.

### 3a. K-means (the method most people know)

K-means picks $k$ centres, assigns each point to its nearest centre, moves each centre to the average of its points, and repeats until nothing changes.

Its problems for our use case:
1. **We must choose $k$** in advance. Here we try $k = 2$ to $15$ and pick the one with the best **silhouette score** (a measure from −1 to 1 of how much closer each point is to its own cluster than to the next one).
2. **Every ETF is forced into a cluster**, even outliers that don't really belong anywhere. That brings in pairs we don't want.
3. **It assumes round, similar-sized clusters.**

In [ ]:
X = features.values

sil = {}
for k in range(2, 16):
    labels_k = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X)
    sil[k] = silhouette_score(X, labels_k)

best_k = max(sil, key=sil.get)
kmeans_labels = KMeans(n_clusters=best_k, n_init=10,
                       random_state=RANDOM_STATE).fit_predict(X)

print("Silhouette score by k:", {k: round(v, 3) for k, v in sil.items()})
print(f"\nBest k = {best_k}. Every one of the {len(X)} ETFs is assigned to a cluster.")

### 3b. OPTICS (the method the paper uses)

The paper lists four things a pairs-trading clustering method should do:

| Requirement | K-means | DBSCAN / OPTICS |
|---|---|---|
| No need to choose the number of clusters | ✗ | ✓ |
| Doesn't force every ETF into a cluster (outliers become **noise**) | ✗ | ✓ |
| Each ETF belongs to exactly one cluster | ✓ | ✓ |
| No assumption about cluster shape | ✗ | ✓ |

**DBSCAN** in one sentence: a cluster is a region where points are packed densely together. It has two settings: $\varepsilon$ ("how close counts as a neighbour") and *minPts* ("how many neighbours make a dense region"). Points in sparse regions are labelled **noise** and never used for pairs.

DBSCAN's weakness: one fixed $\varepsilon$. If one group of ETFs is very tightly packed and another more loosely, no single $\varepsilon$ finds both (Fig. 1 in the paper).

**OPTICS** fixes this. It effectively tries every $\varepsilon$ at once, so we only set *minPts* (`OPTICS_MIN_SAMPLES`). Noise points get the label **−1**.

In [ ]:
optics = OPTICS(min_samples=OPTICS_MIN_SAMPLES).fit(X)
optics_labels = optics.labels_

n_clusters = optics_labels.max() + 1
n_noise = (optics_labels == -1).sum()
print(f"OPTICS found {n_clusters} clusters and left {n_noise} of {len(X)} ETFs as noise.")

**The reachability plot** is the best picture of what OPTICS does. OPTICS visits the ETFs in a special order, and for each one plots how far it is from the dense region before it. **Valleys are clusters** (points close together); **peaks are gaps** between clusters. Notice the valleys can be different depths, which is exactly the "different densities" problem DBSCAN can't handle.

In [ ]:
order = optics.ordering_
reach = optics.reachability_[order]
reach[np.isinf(reach)] = np.nan
lab = optics_labels[order]
tick = features.index[order]

# Alternate dark/light blue between consecutive clusters so you can see where each one starts
cluster_shade, shade, prev = [], 0, None
for l in lab:
    if l != -1 and l != prev:
        shade, prev = 1 - shade, l
    cluster_shade.append(NOISE_COLOR if l == -1 else ("#2a78d6" if shade else "#86b6ef"))

fig, ax = plt.subplots(figsize=(11, 3.8))
ax.bar(range(len(reach)), reach, width=0.85, color=cluster_shade)
ax.set_xticks(range(len(reach)), tick, rotation=90, fontsize=8)
ax.set_ylabel("Reachability distance")
ax.set_title("OPTICS reachability plot: each run of one blue shade is a cluster; grey bars are noise")
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()

## Part 4. What did the clusters find?

### 4a. Picture

Our points live in 5 dimensions, which we can't draw. **t-SNE** squashes them to 2D while trying to keep close points close (the paper does the same, Fig. 8). Only use it for the picture: distances in a t-SNE plot are not exact.

Each dot is an ETF, **coloured by its category**, with the ticker written next to it. Boxes group the ETFs in each cluster. Look for:
- clusters that are all one colour (the method rediscovered a sector on its own), and
- clusters that **mix colours**: relationships a sector-based search would never test. That's the interesting part.

In [ ]:
coords = TSNE(n_components=2, perplexity=min(10, len(X) - 1),
              random_state=RANDOM_STATE, init="pca").fit_transform(X)
coords = pd.DataFrame(coords, index=features.index, columns=["x", "y"])

def cluster_picture(labels, title, ax):
    for cat, col in CATEGORY_COLORS.items():
        pts = coords[categories == cat]
        if len(pts):
            ax.scatter(pts.x, pts.y, s=70, color=col, label=cat,
                       edgecolor="white", linewidth=1.5, zorder=3)
    for t, (x, y) in coords.iterrows():
        ax.annotate(t, (x, y), xytext=(4, 4), textcoords="offset points",
                    fontsize=7, color="#3d3d3a")
    pad = 0.03 * (coords.max() - coords.min()).max()      # box margin: 3% of the plot size
    for c in sorted(set(labels) - {-1}):
        pts = coords[labels == c]
        ax.add_patch(plt.Rectangle((pts.x.min() - pad, pts.y.min() - pad),
                                   pts.x.max() - pts.x.min() + 2 * pad,
                                   pts.y.max() - pts.y.min() + 2 * pad,
                                   fill=False, ec="#5f5e5a", lw=1, ls="--", zorder=1))
        ax.text(pts.x.min() - pad, pts.y.max() + pad, f"C{c}", fontsize=8,
                color="#5f5e5a", va="bottom")
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values(): s.set_visible(False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6.5))
cluster_picture(kmeans_labels, f"K-means (k = {best_k}): every ETF is forced into a box", axes[0])
cluster_picture(optics_labels, "OPTICS: ETFs outside every box are noise", axes[1])
handles, names = axes[0].get_legend_handles_labels()
fig.legend(handles, names, loc="lower center", ncol=len(names), frameon=False, fontsize=9)
plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.show()

### 4b. Cluster members

The table lists every OPTICS cluster with its ETFs and their categories. Questions to answer for your slides:
- **Sanity check:** are GLD and IAU (both physical gold) in the same cluster? SLV and SIVR (both physical silver)? If not, something is wrong.
- Which clusters contain **more than one category**? Can you explain *why* those ETFs move together? (e.g. gold miners vs silver miners, oil producers vs oil services)
- Where did the **outsiders** (SPY, TLT, XLK, ...) go?

In [ ]:
def describe_clusters(labels):
    rows = []
    for c in sorted(set(labels)):
        members = features.index[labels == c]
        rows.append({
            "cluster": "noise" if c == -1 else f"C{c}",
            "size": len(members),
            "categories": ", ".join(sorted(categories[members].unique())),
            "ETFs": ", ".join(members),
        })
    return pd.DataFrame(rows).set_index("cluster")

pd.set_option("display.max_colwidth", None)
print("OPTICS clusters")
describe_clusters(optics_labels)

In [ ]:
print("K-means clusters")
describe_clusters(kmeans_labels)

### 4c. How many pairs would each method test?

This is the paper's Table II. Fewer candidate pairs means fewer chances of a lucky, spurious pass in the cointegration test, *as long as* the good pairs are still in there. Part 5 checks that.

In [ ]:
tickers = list(features.index)

def pairs_within(groups):
    # all pairs of ETFs that share a group label (noise label -1 never forms pairs)
    out = []
    for g in set(groups) - {-1}:
        members = [t for t, gi in zip(tickers, groups) if gi == g]
        out += list(combinations(sorted(members), 2))
    return out

candidates = {
    "All pairs":     list(combinations(sorted(tickers), 2)),
    "Same category": pairs_within(list(categories[tickers])),
    "K-means":       pairs_within(list(kmeans_labels)),
    "OPTICS":        pairs_within(list(optics_labels)),
}

summary = pd.DataFrame({
    "groups": [1, categories.nunique(), best_k, n_clusters],
    "ETFs used": [len(tickers), len(tickers), len(tickers), len(tickers) - n_noise],
    "candidate pairs": [len(v) for v in candidates.values()],
}, index=candidates.keys())
summary

**End of Sunday.** Screenshot the reachability plot, the t-SNE picture and the two tables: they're slide material. Write down one or two clusters you can explain in plain English.

---
## Part 5 (Tuesday). Filtering the pairs

Being in the same cluster only means two ETFs *react to the same factors*. It doesn't mean their spread will mean-revert in a tradeable way. Following the paper (Section III-C), a pair is kept only if it passes **all four** tests, run on the formation window only:

| # | Test | Why |
|---|---|---|
| 1 | **Cointegrated** (Engle-Granger p-value < 0.05) | The spread must return to a mean. Same `coint()` as the pairs trading notebook. The paper runs it **both ways round** (Y on X and X on Y) and keeps the stronger result, because the test depends on which asset you put first. |
| 2 | **Hurst exponent < 0.5** | A second, independent check of mean reversion. $H < 0.5$: mean-reverting. $H = 0.5$: random walk. $H > 0.5$: trending. |
| 3 | **Half-life between 1 day and 1 year** | Same half-life as the pairs trading notebook. Too fast and trading costs eat the profit; too slow and money sits in a trade for ages. |
| 4 | **Crosses its mean at least 12 times a year** | About one trading opportunity per month. A spread that reverts but rarely crosses gives too few trades. |

The tests use **log prices**, like the pairs trading notebook.

In [ ]:
from statsmodels.tsa.stattools import coint

log_form = np.log(prices.loc[form_slice])
YEARS = len(log_form) / 252

def hurst(series, max_lag=100):
    # How fast does the typical change grow with the time gap?
    # Random walk: std of an n-day change grows like n^0.5 -> H = 0.5.
    # Mean-reverting: grows more slowly -> H < 0.5.
    lags = np.arange(2, max_lag)
    tau = [np.std(series[lag:] - series[:-lag]) for lag in lags]
    return np.polyfit(np.log(lags), np.log(tau), 1)[0]

def half_life(spread):
    # Same OU regression as the pairs trading notebook: s_{t+1} = a + b s_t
    b, a = np.polyfit(spread[:-1], spread[1:], 1)
    return -np.log(2) / np.log(b) if 0 < b < 1 else np.inf    # in trading days

def test_pair(t1, t2):
    p1, p2 = log_form[t1].values, log_form[t2].values
    # 1. Engle-Granger both ways round; keep the more negative t-statistic
    stat_a, p_a, _ = coint(p1, p2)
    stat_b, p_b, _ = coint(p2, p1)
    if stat_a <= stat_b:
        y_t, x_t, p_value = t1, t2, p_a
    else:
        y_t, x_t, p_value = t2, t1, p_b
    y, x = log_form[y_t].values, log_form[x_t].values
    beta1, beta0 = np.polyfit(x, y, 1)
    s = y - beta0 - beta1 * x
    crossings = np.sum(np.diff(np.sign(s - s.mean())) != 0) / YEARS
    return {"y": y_t, "x": x_t, "beta0": beta0, "beta1": beta1,
            "p_value": p_value, "hurst": hurst(s), "half_life": half_life(s),
            "crossings_per_year": crossings}

# Test every candidate pair once (the "All pairs" set contains all the others)
tested = pd.DataFrame([dict(pair=pr, **test_pair(*pr)) for pr in candidates["All pairs"]]).set_index("pair")

tested["passes"] = ((tested.p_value < 0.05) & (tested.hurst < 0.5)
                    & tested.half_life.between(1, 252) & (tested.crossings_per_year >= 12))
print(f"Tested {len(tested)} pairs, {tested.passes.sum()} pass all four filters.")

In [ ]:
rows = {}
for method, prs in candidates.items():
    t = tested.loc[prs] if prs else tested.iloc[:0]
    rows[method] = {
        "candidate pairs": len(t),
        "1. cointegrated": int((t.p_value < 0.05).sum()),
        "selected (all 4)": int(t.passes.sum()),
        "selected / candidates": f"{t.passes.mean():.1%}" if len(t) else "-",
    }
pd.DataFrame(rows).T

**Questions to answer:**
- What share of *All pairs* gets selected compared with *OPTICS*? A higher hit rate inside clusters suggests clustering is pointing us at real relationships.
- Did GLD/IAU survive? Duplicates usually fail the **half-life** filter: their spread is tiny noise that "reverts" in under a day, so there's nothing to trade after costs. If they pass, remove them by hand and say why on a slide.
- Which selected pairs are **cross-category**? Those are pairs a sector-based search would have missed.

Next we save the selected pairs. Friday's backtest reads this file and trades them on 2023 (validation) and 2024 (test).

In [ ]:
selected = []
for method, prs in candidates.items():
    t = tested.loc[prs] if prs else tested.iloc[:0]
    t = t[t.passes].reset_index().assign(method=method)
    selected.append(t)
selected = pd.concat(selected, ignore_index=True)
selected["category_y"] = selected.y.map(UNIVERSE)
selected["category_x"] = selected.x.map(UNIVERSE)
selected["cross_category"] = selected.category_y != selected.category_x

selected.drop(columns="pair").to_csv("selected_pairs.csv", index=False)
print(f"Saved {len(selected)} rows to selected_pairs.csv")
selected[selected.method == "OPTICS"].drop(columns=["pair", "method"]).round(3)